# FHIR R4 → OMOP CDM v5.5 via ViewDefinitions

The `fhir4ds.viewdef` OMOP mapping library flattens FHIR resources into
OMOP-shaped views. Concepts resolve through a ConceptMap RESOURCE join
(runtime join, `system + concept_code` keys); unmapped codes keep their
source values with concept_id 0.

This notebook runs on the real sample data: a 25-patient Synthea R4
cohort plus the RxNorm ConceptMap, hosted on Hugging Face (public,
CORS-enabled — the same data the Viewing Room loads in the browser).

In [1]:
# Install the unified package (PyPI).
# For a local wheel build instead:
#   %pip install -q --find-links "file:///abs/path/to/fhir4ds/dist" fhir4ds-v2==0.0.19
%pip install -q fhir4ds-v2==0.0.19


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import urllib.request

import fhir4ds
from fhir4ds.viewdef.omop_runner import create_omop_views

DATA = "https://huggingface.co/datasets/fhir4ds/omop-demo/resolve/main"

def fetch(path: str) -> list:
    with urllib.request.urlopen(f"{DATA}/{path}") as resp:
        text = resp.read().decode("utf-8")
    return [line for line in text.splitlines() if line.strip()]

con = fhir4ds.create_connection()
con.execute("CREATE TABLE resources (resource JSON)")

# Patients + encounters (includes the curated ConceptMap supplements).
for line in fetch("encounter-resources.ndjson"):
    con.execute("INSERT INTO resources VALUES (?)", [line])

print(con.execute("SELECT count(*) FROM resources").fetchone()[0], "resources staged")

1310 resources staged


### Compose the OMOP schema

`create_omop_views` materializes the library views (dependency order
inferred from `Join.view` targets), composes the unions, and applies the
integer-key + concept-0 projections.

In [3]:
created = create_omop_views(con, source_table="resources")
print(sorted(created))

['__join_targets__', 'condition_occurrence', 'death', 'device_exposure', 'drug_exposure', 'measurement', 'observation', 'person', 'procedure_occurrence', 'visit_occurrence']


In [4]:
con.execute("SELECT person_id, gender_concept_id, year_of_birth, person_source_value FROM omop.person").df()

,person_id,gender_concept_id,year_of_birth,person_source_value
0,5836428378447210269,8532,1957,e61b54c7-8ae8-907c-4e6d-20f5e86eca63
1,6612219856025284181,8507,1972,4e3be038-c545-8835-4859-86cb59cee238
2,5426067500172554819,8532,1998,7a4ff83b-37bc-fe9a-b385-d2615aeac488
3,3169254208758680576,8507,2004,431ddba2-3b8f-c671-ffa3-c1d174182212
4,7789657465346563585,8532,1967,42c3356a-3949-19d3-39a0-256d5d7b4203
5,59738942610178675,8507,1972,6c519982-779f-dc2b-649b-03961010a799
6,8631096884550548484,8532,2009,f4d451dd-b975-32d1-3c43-80aea228044d
7,4861816556410089142,8507,1964,010b7799-d5b3-3cba-b98e-ca9051c68012
8,8811488753653818170,8532,1972,12d453f1-7887-9625-b10d-9b3609caff6d
9,1615202078500127447,8507,1971,c185cdd3-2c37-4f65-512d-d25cd62d3904


In [5]:
con.execute("""
SELECT visit_concept_id, count(*) AS visits
FROM omop.visit_occurrence GROUP BY 1 ORDER BY 2 DESC
""").df()

,visit_concept_id,visits
0,9202,1212
1,9203,43
2,9201,22
3,0,7


### Medications: the cross-resource union

`drug_exposure` spans MedicationRequest / Dispense / Statement /
Administration / Immunization — a `unionAll` across the child views,
which a single ViewDefinition cannot express. Stage the medication
bundle (patients + medications + immunizations + the chunked RxNorm
ConceptMap) and re-compose.

In [6]:
con.execute("DELETE FROM resources")
for line in fetch("medication-resources.ndjson"):
    con.execute("INSERT INTO resources VALUES (?)", [line])
create_omop_views(con, source_table="resources")

con.execute("""
SELECT src, count(*) AS rows, count(*) FILTER (WHERE drug_concept_id <> 0) AS resolved
FROM omop.drug_exposure GROUP BY 1 ORDER BY 2 DESC
""").df()

,src,rows,resolved
0,MedicationRequest,1170,1024
1,Immunization,318,0


In [7]:
con.execute("""
SELECT drug_concept_id, drug_source_value, drug_exposure_start_date, src
FROM omop.drug_exposure WHERE drug_concept_id <> 0 LIMIT 8
""").df()

,drug_concept_id,drug_source_value,drug_exposure_start_date,src
0,1836434,198335,2021-07-07,MedicationRequest
1,19125062,665078,2010-03-28,MedicationRequest
2,19125062,665078,1995-12-30,MedicationRequest
3,19132594,808917,2020-01-22,MedicationRequest
4,40232448,1049630,1968-06-22,MedicationRequest
5,40232448,1049630,1981-11-08,MedicationRequest
6,19019979,198405,2020-09-07,MedicationRequest
7,1713671,562251,2025-03-12,MedicationRequest


### Notes

- Concepts resolve at query time through the ConceptMap view join;
  restage a different ConceptMap to swap vocabularies without touching
  any ViewDefinition.
- ConceptMap resources are chunked (20 elements each) because DuckDB
  cannot unnest monolithic 100k-element JSON arrays.
- The full CDM surface (measurement/observation/death and the schema,
  required-field, grain, mapping and vd-conformance sections) is covered
  by `conformance/scripts/run_omop.py`.